# Cross-Participant MEG Faces Decoding

This is a companion tutorial to the paper *"A Primer on Low-Dimensional Neural Dynamics: PCA-Based Trajectory Analysis for EEG and MEG."*

The MEG trajectory tutorial showed that famous, unfamiliar and scrambled faces trace measurably different paths through a low-dimensional state space, with the face/scrambled distinction peaking near the N170. This notebook asks the harder, predictive version of that question:

> **Can face category be decoded on single trials in a participant whose labeled trials were never used for training?**

Behind it sits the question that decides whether the low-dimensional framing earns its place:

> **Can a 30-dimensional trajectory beat 306 sensors?**

The honest answer, stated up front, is that it cannot beat them at pure feature quality — a PCA is a linear projection of the sensors, so a 30-component trajectory is a rotation and truncation of the 306 channels and can contain less information, never more. What it *can* do is something the sensor array structurally cannot: because a trajectory is a **path through a latent space**, paths from different participants can be rotated into correspondence with each other. Maxwell filtering already puts every head into a common physical frame, and that physical correspondence is exactly what fails to transfer. This notebook tests whether latent-path correspondence transfers better.

<div class="alert alert-secondary">
<b>🗺️ Analysis roadmap:</b><br>
<ol style="margin-bottom: 0; margin-top: 5px;">
  <li>Define the predictive question and the inferential unit.</li>
  <li>Load the whitened sensor-time representation.</li>
  <li>Verify target encoding and participant-level class balance.</li>
  <li>Declare leave-one-participant-out cross-validation.</li>
  <li>Configure fold-local sliding temporal decoding.</li>
  <li>Build five representations of the same data.</li>
  <li>Run and audit every outer fold.</li>
  <li>Inspect time-resolved generalization and gain through time.</li>
  <li>Examine held-out-participant variability.</li>
  <li>Test the differences with a paired permutation test.</li>
  <li>Bound the alignment result with three validations.</li>
  <li>Export the full analysis and structured HTML report.</li>
</ol>
</div>

### The five representations

Every representation feeds the *identical* classifier, folds, scaling, metric and time axis. Only the feature space changes, which is what makes the comparison interpretable.

| Representation | Basis | Per-participant? | Rotated onto a shared template? |
|---|---|---|---|
| **Sensors** | 306 whitened MEG channels | — | — |
| **Shared PCA** | one PCA on pooled training participants | no | — |
| **Unaligned PCA** | one PCA per participant | yes | **no** |
| **Aligned PCA (3C / 30C)** | one PCA per participant | yes | **yes** |

*Unaligned PCA* is the control that decides the story. It differs from *Aligned PCA* by exactly one operation — the Procrustes rotation — so a difference between them is attributable to the rotation rather than to the mere fact of having a participant-specific basis.

## 0. Setup & Configuration

Every scientific choice is declared before a single trial is loaded. The notebook performs the complete analysis in its own cells; only the report renderer is shared with the companion script, so both entry points produce exactly the same structured HTML output.

### 0.1. Environment & imports

`coco-pipe` owns the cross-validation engine, the fold-local preprocessing, the temporal estimator, the alignment transformer, the result containers, the statistics and the plotting. The repository helpers only load the prepared Wakeman–Henson epochs and record provenance.

In [1]:
# --- Standard library -------------------------------------------------------
import os
import warnings
from pathlib import Path

# --- Numerical and plotting libraries ---------------------------------------
import numpy as np
import pandas as pd

# --- coco-pipe decoding ------------------------------------------------------
from coco_pipe.decoding import (
    ChanceAssessmentConfig,
    CVConfig,
    Experiment,
    ExperimentConfig,
    ReducerConfig,
    StatisticalAssessmentConfig,
    TemporalAlignmentConfig,
    TemporalDecoderConfig,
)
from coco_pipe.decoding.configs import ClassicalModelConfig
from coco_pipe.decoding.stats import run_paired_permutation_assessment
from coco_pipe.transforms import TemporalProcrustesAlignment
from coco_pipe.viz.interactive import (
    plot_distribution_groups,
    plot_group_scatter_with_mean,
    plot_heatmap,
)
from coco_pipe.viz.interactive.decoding import (
    plot_temporal_score_curve,
    plot_temporal_statistical_assessment,
)
from coco_pipe.viz.theme import set_coco_theme

# --- Repository helpers ------------------------------------------------------
from pca_neural_trajectories import (
    contrast_label,
    facet_figures,
    write_manifest,
)
from pca_neural_trajectories.wakeman_henson import (
    LABEL_NAMES,
    MEG_SENSOR_SETS,
    _load_wakeman_henson_container,
)

### 0.2. Analysis parameters & output contract

The default contrast is Famous versus Scrambled — a perceptual categorisation with a large, early, well-localized evoked signature. `CONDITIONS` accepts any tuple of condition ids (`1 = Famous`, `2 = Unfamiliar`, `3 = Scrambled`); two give binary decoding, three give multiclass, and chance moves accordingly.

`SENSOR_SET` optionally restricts the analysis to the official VectorView occipital, temporal or combined helmet selections. These are **sensor positions, not source-localized cortical ROIs** — a temporal-sensor result is a statement about where the field was strongest outside the head, not about which gyrus produced it.

<div class="alert alert-info">
<b>⚙️ Environment overrides:</b><br>
Use <code>MEG_DECODING_N_JOBS</code>, <code>MEG_DECODING_PERMUTATIONS</code>, <code>MEG_DERIVATIVES_ROOT</code> and <code>MEG_DECODING_OUTPUT</code> to change runtime settings without editing analysis cells.
</div>

In [2]:
set_coco_theme(mode="paper", colorblind=True)

SEED = 42
CONDITIONS = (1, 3)                    # 1 = Famous, 2 = Unfamiliar, 3 = Scrambled
CHANCE_LEVEL = 1.0 / len(CONDITIONS)
COMPARISON_CONTRASTS = ((1, 3), (2, 3), (1, 2), (1, 2, 3))
# Famous/Unfamiliar vs Scrambled, familiarity, and three-class
RUN_CONTRAST_COMPARISON = os.getenv("MEG_CONTRAST_COMPARISON", "1") == "1"
SENSOR_SET = os.getenv("MEG_SENSOR_SET", "all_sensors")
N_COMPONENTS = int(os.getenv("MEG_DECODING_COMPONENTS", "30"))
SMALL_N_COMPONENTS = int(os.getenv("MEG_DECODING_SMALL_COMPONENTS", "3"))
N_PERMUTATIONS = int(os.getenv("MEG_DECODING_PERMUTATIONS", "200"))
WITHIN_SUBJECT_SPLITS = 5
GEOMETRY_MAX_TRIALS = 200
N_JOBS = int(os.getenv("MEG_DECODING_N_JOBS", "1"))
N_SUBJECTS = int(os.getenv("MEG_N_SUBJECTS", "3"))
SUBJECTS = tuple(f"{subject:02d}" for subject in range(1, N_SUBJECTS + 1))
DERIVATIVES_ROOT = Path(
    os.getenv(
        "MEG_DERIVATIVES_ROOT",
        str(Path.home() / "mne_data" / "ds000117" / "derivatives" / "pca_trajectories"),
    )
)
OUTPUT = Path(os.getenv("MEG_DECODING_OUTPUT", "outputs/tutorial_megfaces_decoding")) / SENSOR_SET
FIGURES_DIR = OUTPUT / "figures"
RESULTS_DIR = OUTPUT / "experiment_results"

if SENSOR_SET not in MEG_SENSOR_SETS:
    raise ValueError(f"SENSOR_SET must be one of {tuple(MEG_SENSOR_SETS)}.")

SHARED_PCA = f"Shared PCA ({N_COMPONENTS})"
UNALIGNED_PCA = f"Unaligned PCA ({N_COMPONENTS}C)"
ALIGNED_SMALL = f"Aligned PCA ({SMALL_N_COMPONENTS}C)"
ALIGNED_LARGE = f"Aligned PCA ({N_COMPONENTS}C)"
CALIBRATION = f"Aligned PCA ({N_COMPONENTS}C, calibration)"
REPRESENTATION_COLORS = {
    "Sensors": "#1b9e77",
    SHARED_PCA: "#e6ab02",
    UNALIGNED_PCA: "#d95f02",
    ALIGNED_SMALL: "#7570b3",
    ALIGNED_LARGE: "#2a78d6",
    CALIBRATION: "#c51b7d",
    "Within participant (sensors)": "#444444",
}

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Participants requested: {SUBJECTS}")
print(f"Derivatives:   {DERIVATIVES_ROOT}")
print(f"Output bundle: {OUTPUT}")

Participants requested: ('01', '02', '03')
Derivatives:   /Users/hamzaabdelhedi/mne_data/ds000117/derivatives/pca_trajectories
Output bundle: outputs/tutorial_megfaces_decoding/all_sensors


## Step 1. Define the Predictive Question

The question is **cross-participant generalization**, not whether a model can classify held-out trials from a familiar participant. That distinction determines the entire validation design.

- **Observation:** one epoched image-presentation trial.
- **Target:** image category, `0 = Famous`, `1 = Scrambled` by default.
- **Inferential unit:** the participant.
- **Generalization target:** a participant whose labeled trials were entirely absent during training.

<div class="alert alert-danger">
<b>🚫 Why random trial splitting is invalid here:</b><br>
Trials from one participant share head geometry, head position in the dewar, sensor noise, SSS reconstruction and stable idiosyncratic response shape. If some of a participant's trials are in training and others in test, the classifier can score well by recognizing <i>the participant</i>. The reported number would then answer a question nobody asked.
</div>

## Step 2. Load the Whitened Sensor-Time Representation

The prepared tensor loads as `trial × sensor × time`. Each participant has already been whitened with their own empty-room covariance, which places magnetometers (in T) and gradiometers (in T/m) on a common noise scale — without it, one sensor type's raw units would silently dominate any variance-based decomposition.

Note what is *not* done here: no global standardization, no global PCA, no channel selection. Anything fitted on data is fitted later, inside a fold, after the participants have been separated.

<div class="alert alert-warning">
<b>🧠 Preprocessing boundary:</b><br>
The MEG main-analysis tutorial prepares these derivatives (Maxwell filtering with movement compensation, per-participant autoreject thresholds, empty-room whitening). This notebook loads them and does not preprocess.
</div>

In [3]:
if not DERIVATIVES_ROOT.exists():
    raise FileNotFoundError(
        f"Prepared MEG derivatives were not found at {DERIVATIVES_ROOT}. "
        "Run the MEG main-analysis tutorial or its preparation script first."
    )

container = _load_wakeman_henson_container(
    DERIVATIVES_ROOT,
    subjects=SUBJECTS,
    conditions=CONDITIONS,
    sensor_set=SENSOR_SET,
)
X = np.asarray(container.X, dtype=np.float32)
times = np.asarray(container.coords["time"], dtype=float)
condition = np.asarray(container.y, dtype=int)
sensor_names = np.asarray(container.coords["channel"]).astype(str)
subject_ids = np.asarray(container.coords["subject"]).astype(str)
trial_ids = (
    np.asarray(container.ids).astype(str)
    if container.ids is not None
    else np.asarray([f"trial-{index}" for index in range(len(X))])
)
WHITENING = container.meta.get("whitening", "not recorded")
del container

condition_to_class = {value: index for index, value in enumerate(CONDITIONS)}
y = np.array([condition_to_class[value] for value in condition])
TARGET_NAMES = [LABEL_NAMES[value] for value in CONDITIONS]

ANALYZED_SUBJECTS = sorted(np.unique(subject_ids).tolist())
if len(ANALYZED_SUBJECTS) < 3:
    raise RuntimeError("Cross-participant decoding needs at least three participants.")
if N_COMPONENTS > X.shape[1]:
    raise ValueError(f"N_COMPONENTS={N_COMPONENTS} exceeds {X.shape[1]} sensors.")

Reading /Users/hamzaabdelhedi/mne_data/ds000117/derivatives/pca_trajectories/sub-01/ses-meg/meg/sub-01_ses-meg_task-facerecognition_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...     800.00 ms
        0 CTF compensation matrices available
Adding metadata with 6 columns
886 matching events found
No baseline correction applied
0 projection items activated


/Users/hamzaabdelhedi/Projects/research/pca-neural-trajectories-eeg-meg/pca_neural_trajectories/wakeman_henson.py:660: RuntimeWarning: Something went wrong in the data-driven estimation of the data rank as it exceeds the theoretical rank from the info (74 > 72). Consider setting rank to "auto" or setting it explicitly as an integer.
  whitener, ch_names = mne.cov.compute_whitener(


Reading /Users/hamzaabdelhedi/mne_data/ds000117/derivatives/pca_trajectories/sub-02/ses-meg/meg/sub-02_ses-meg_task-facerecognition_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...     800.00 ms
        0 CTF compensation matrices available
Adding metadata with 6 columns
881 matching events found
No baseline correction applied
0 projection items activated


/Users/hamzaabdelhedi/Projects/research/pca-neural-trajectories-eeg-meg/pca_neural_trajectories/wakeman_henson.py:660: RuntimeWarning: Something went wrong in the data-driven estimation of the data rank as it exceeds the theoretical rank from the info (73 > 72). Consider setting rank to "auto" or setting it explicitly as an integer.
  whitener, ch_names = mne.cov.compute_whitener(


Reading /Users/hamzaabdelhedi/mne_data/ds000117/derivatives/pca_trajectories/sub-03/ses-meg/meg/sub-03_ses-meg_task-facerecognition_epo.fif ...
    Found the data of interest:
        t =    -200.00 ...     800.00 ms
        0 CTF compensation matrices available
Adding metadata with 6 columns
585 matching events found
No baseline correction applied
0 projection items activated


/Users/hamzaabdelhedi/Projects/research/pca-neural-trajectories-eeg-meg/pca_neural_trajectories/wakeman_henson.py:660: RuntimeWarning: Something went wrong in the data-driven estimation of the data rank as it exceeds the theoretical rank from the info (72 > 71). Consider setting rank to "auto" or setting it explicitly as an integer.
  whitener, ch_names = mne.cov.compute_whitener(


In [4]:
data_summary = pd.Series({
    "trials": X.shape[0],
    "sensors": X.shape[1],
    "time samples": X.shape[2],
    "participants": len(ANALYZED_SUBJECTS),
    "first time (s)": float(times[0]),
    "last time (s)": float(times[-1]),
    "sensor set": SENSOR_SET,
    "whitening": WHITENING,
    "target": " vs ".join(TARGET_NAMES),
})
data_summary.to_frame("value")

,value
trials,1565
sensors,306
time samples,251
participants,3
first time (s),-0.2
last time (s),0.8
sensor set,all_sensors
whitening,subject-wise empty-room covariance
target,Famous vs Scrambled


## Step 3. Verify the Target and Participant-Level Class Balance

Class index `i` always corresponds to `CONDITIONS[i]`, so the mapping is fixed before any model sees the data. We then document the number of trials in every participant × class cell.

Balanced accuracy is the mean recall across classes, so each class contributes equally even when a participant has unequal trial counts — which they will, because per-participant autoreject thresholds discard different numbers of trials in different heads. The classifier additionally receives `class_weight='balanced'`, estimated from the **training fold only**.

<div class="alert alert-success">
<b>✅ No test-set resampling:</b><br>
Class balance is handled by the metric and by training-fold weights. Test trials are never duplicated, removed, or used to choose a training weight.
</div>

In [5]:
trial_counts = (
    pd.DataFrame({
        "subject": subject_ids,
        "condition": condition,
        "condition_name": [LABEL_NAMES[value] for value in condition],
    })
    .groupby(["subject", "condition", "condition_name"], as_index=False)
    .size()
    .rename(columns={"size": "n_trials"})
)
trial_counts.pivot(index="subject", columns="condition_name", values="n_trials")

condition_name,Famous,Scrambled
subject,,
01,294,297
02,293,292
03,206,183


## Step 4. Declare Leave-One-Participant-Out Cross-Validation

`CVConfig(strategy='leave_one_group_out')` creates one outer fold per participant. Every trial from the held-out participant becomes test data; every trial from all other participants becomes training data.

The same participant groups are passed to every representation, so the resulting scores are **paired by held-out participant** — the property that later lets us test differences rather than merely eyeball two curves. We verify the generated splits after fitting rather than trusting that the configuration was applied as intended.

## Step 5. Configure Fold-Local Sliding Temporal Decoding

At each latency, `wrapper='sliding'` fits a separate logistic-regression classifier on the feature values at that one time point. The result is a time-resolved generalization curve rather than a single number, which lets us ask *when* the discriminative information appears — and for faces, whether it appears at the N170.

`use_scaler=True` instructs `coco-pipe` to fit standardization inside each training fold; the held-out participant is transformed with those training-fold parameters. Fitting a single `StandardScaler` on all participants before splitting would quietly leak the test distribution into the training pipeline.

<div class="alert alert-warning">
<b>📈 Multiple-latency caution:</b><br>
Every plotted time point is a separately fitted model. The highest point is a useful descriptive summary, but it is not automatically a family-wise-error-corrected result. Step 10 supplies the corrected test.
</div>

In [6]:
decoder = TemporalDecoderConfig(
    wrapper="sliding",
    base=ClassicalModelConfig(
        estimator="LogisticRegression",
        params={"class_weight": "balanced", "max_iter": 2000},
    ),
    n_jobs=1,
    verbose=False,
)
sensor_config = ExperimentConfig(
    task="classification",
    models={"Logistic regression": decoder},
    metrics=["balanced_accuracy"],
    cv=CVConfig(strategy="leave_one_group_out", shuffle=False),
    use_scaler=True,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=False,
)
sensor_config

ExperimentConfig(task='classification', output_dir=None, tag='experiment', random_state=42, models={'Logistic regression': TemporalDecoderConfig(kind='temporal', wrapper='sliding', base=ClassicalModelConfig(kind='classical', method='ClassicalModel', estimator='LogisticRegression', params={'class_weight': 'balanced', 'max_iter': 2000}, input_kind='tabular'), scoring=None, n_jobs=1, position=0, allow_2d=False, verbose=False)}, grids=None, cv=CVConfig(strategy='leave_one_group_out', n_splits=5, shuffle=False, random_state=42, test_size=0.2, stratify=False, group_key=None, auto_reduce_n_splits=True), tuning=TuningConfig(enabled=False, search_type='grid', n_iter=10, scoring=None, n_jobs=-1, random_state=42, cv=None, allow_nongroup_inner_cv=False), feature_selection=FeatureSelectionConfig(enabled=False, method='sfs', n_features=None, direction='forward', tol=None, cv=None, scoring=None, allow_nongroup_inner_cv=False), erasure=ErasureConfig(enabled=False, method='leace', params={}), temporal_

## Step 6. Build Five Representations of the Same Data

Each representation copies the sensor configuration and changes exactly one field, so the feature space is the only experimental variable.

<span style="color: #0072B2;"><b>Shared PCA</b></span> — a single fold-local PCA fitted on the pooled training participants (`ReducerConfig`) and applied unchanged to everyone, including the held-out participant. There is no per-participant basis and no rotation. This is the classic "reduce dimensionality, then decode" baseline, and it is the one that tests the *compression* claim: does a 30-dimensional summary keep what 306 sensors had?

<span style="color: #0072B2;"><b>Unaligned PCA</b></span> — a separate PCA per participant, and nothing else. Each participant is described in their *own* principal axes. Since PCA orders components by variance and the variance ordering differs between heads, participant 3's "PC1" and participant 5's "PC1" need not be the same functional direction at all. This is the control that decides the whole story.

<span style="color: #0072B2;"><b>Aligned PCA</b></span> — *temporal-shape alignment*, constructed for exactly this problem; its closest relative is hyperalignment, adapted from static response patterns to trajectories. Inside each fold:

1. fit a shared PCA on the training participants, project their trials, and average over **all** trials ignoring condition — producing a `(time × K)` template path;
2. fit a separate PCA on each participant's own data, held-out participant included;
3. Procrustes-rotate each participant's own grand-mean trajectory onto the template, and apply that rotation to all of their trials.

Every participant then lives in a frame where their average response path points the same way as the group's. The target being matched is a **path through a latent state space** — an object that only exists once you have reduced dimensionality, which is why this operation has no direct sensor-space equivalent. (Euclidean Alignment and Riemannian Procrustes Analysis do align sensor-space *distributions* across participants; matching a time-resolved trajectory *shape* is a different object, and until those run as baselines on the same folds any gain here is a gain over an **unaligned** sensor decoder.)

Note that the template is built without ever looking at a condition label, which is why the alignment can legitimately be estimated for a participant whose labels we are withholding.

We run alignment at two component counts: three, the number the trajectory figures live in, and thirty, closer to the effective dimensionality of 306-channel MEG.

<div class="alert alert-danger">
<b>🔒 Transductive scope:</b><br>
Aligned performance assumes an <i>unlabeled calibration batch</i> from the new participant — no labels are used, but some of their data is. This is standard domain adaptation and realistic in practice, but it is not "train on 15 participants and apply blind to a single trial from the 16th". Step 11 quantifies exactly how much of the result depends on this.
</div>

In [7]:
shared_pca_config = sensor_config.model_copy(deep=True)
shared_pca_config.reducer = ReducerConfig(enabled=True, n_components=N_COMPONENTS)

# The control: identical per-participant PCA, with the rotation switched off.
unaligned_config = sensor_config.model_copy(deep=True)
unaligned_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=N_COMPONENTS, adaptation="transductive", rotate=False
)

aligned_small_config = sensor_config.model_copy(deep=True)
aligned_small_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=SMALL_N_COMPONENTS, adaptation="transductive"
)

aligned_large_config = sensor_config.model_copy(deep=True)
aligned_large_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=N_COMPONENTS, adaptation="transductive"
)

# Validation variant (Step 11): each held-out participant's rotation is estimated
# from one half of their trials and applied to the other half.
calibration_config = sensor_config.model_copy(deep=True)
calibration_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True, n_components=N_COMPONENTS, adaptation="calibration"
)

experiments = {
    "Sensors": sensor_config,
    SHARED_PCA: shared_pca_config,
    UNALIGNED_PCA: unaligned_config,
    ALIGNED_SMALL: aligned_small_config,
    ALIGNED_LARGE: aligned_large_config,
}
REPRESENTATION_NAMES = list(experiments)
OTHER_REPRESENTATIONS = [name for name in REPRESENTATION_NAMES if name != "Sensors"]

pd.DataFrame([
    {
        "Representation": name,
        "shared_pca_components": (
            config.reducer.n_components if config.reducer.enabled else None
        ),
        "subject_pca_components": (
            config.temporal_alignment.n_components
            if config.temporal_alignment.enabled else None
        ),
        "rotated": (
            config.temporal_alignment.rotate
            if config.temporal_alignment.enabled else None
        ),
        "adaptation": (
            config.temporal_alignment.adaptation
            if config.temporal_alignment.enabled else None
        ),
        "scaler_inside_fold": config.use_scaler,
    }
    for name, config in {**experiments, CALIBRATION: calibration_config}.items()
])

,Representation,shared_pca_components,subject_pca_components,rotated,adaptation,scaler_inside_fold
0,Sensors,NaN,NaN,None,NaN,True
1,Shared PCA (30),30.0,NaN,None,NaN,True
2,Unaligned PCA (30C),NaN,30.0,False,transductive,True
3,Aligned PCA (3C),NaN,3.0,True,transductive,True
4,Aligned PCA (30C),NaN,30.0,True,transductive,True
5,"Aligned PCA (30C, calibration)",NaN,30.0,True,calibration,True


## Step 7. Run and Preserve Every Outer Fold

All six experiments receive the same epochs, labels, participant groups, trial identifiers and scientific time axis. `ExperimentResult` retains the fold scores, per-trial predictions, splits, fit diagnostics and configuration needed for a later audit.

We export the complete result objects rather than only the final mean curve. That is what makes Step 10 cheap: the permutation test re-scores stored predictions instead of refitting thousands of temporal classifiers.

In [8]:
results = {}
temporal_frames = []
fold_frames = []
diagnostic_frames = []

for representation, config in {**experiments, CALIBRATION: calibration_config}.items():
    print(f"Running {representation} ...")
    result = Experiment(config).run(
        X,
        y,
        groups=subject_ids,
        sample_ids=trial_ids,
        observation_level="epoch",
        inferential_unit="subject",
        time_axis=times,
    )
    results[representation] = result

    temporal = result.get_temporal_score_summary()
    temporal["Estimator"] = temporal["Model"]
    temporal["Representation"] = representation
    temporal["Model"] = representation
    temporal_frames.append(temporal)

    folds = result.get_detailed_scores()
    folds["Estimator"] = folds["Model"]
    folds["Representation"] = representation
    fold_frames.append(folds)

    diagnostics = result.get_fit_diagnostics()
    diagnostics["Estimator"] = diagnostics["Model"]
    diagnostics["Representation"] = representation
    diagnostic_frames.append(diagnostics)

    slug = "".join(c if c.isalnum() else "_" for c in representation.lower()).strip("_")
    result.export(RESULTS_DIR / slug, config=config.model_dump(), formats=("csv",))

temporal_scores = pd.concat(temporal_frames, ignore_index=True)
fold_scores = pd.concat(fold_frames, ignore_index=True)
fit_diagnostics = pd.concat(diagnostic_frames, ignore_index=True)

Running Sensors ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Shared PCA (30) ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Unaligned PCA (30C) ...


...

Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Aligned PCA (30C) ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


Running Aligned PCA (30C, calibration) ...


Lightweight assessment failed for Logistic regression: No scalar predictions found for post-hoc assessment.


### Audit the generated splits

Configuration is not proof. We derive a fold audit from the stored splits and verify that every fold contains exactly one test participant, with no participant appearing in both training and test. Every representation uses the identical outer splitter, so auditing one is enough.

In [9]:
split_rows = results["Sensors"].get_splits()
audit_records = []
for fold in sorted(split_rows["Fold"].unique()):
    fold_rows = split_rows[split_rows["Fold"] == fold]
    train_rows = fold_rows[fold_rows["Set"] == "train"]
    test_rows = fold_rows[fold_rows["Set"] == "test"]
    train_subjects = sorted(train_rows["Group"].astype(str).unique())
    test_subjects = sorted(test_rows["Group"].astype(str).unique())
    overlap = sorted(set(train_subjects) & set(test_subjects))
    audit_records.append({
        "Fold": int(fold),
        "held_out_subject": ", ".join(test_subjects),
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_train_trials": len(train_rows),
        "n_test_trials": len(test_rows),
        "subject_overlap": ", ".join(overlap),
        "leakage_free": len(overlap) == 0 and len(test_subjects) == 1,
    })

split_audit = pd.DataFrame(audit_records)
if not split_audit["leakage_free"].all():
    raise RuntimeError("The LOSO audit found participant overlap.")
split_audit

,Fold,held_out_subject,n_train_subjects,n_test_subjects,n_train_trials,n_test_trials,subject_overlap,leakage_free
0,0,01,2,1,974,591,,True
1,1,02,2,1,980,585,,True
2,2,03,2,1,1176,389,,True


## Step 8. Inspect Time-Resolved Cross-Participant Generalization

The line is mean balanced accuracy across held-out participants; the ribbon is the fold-level standard deviation. The dotted horizontal line marks chance and the vertical line marks image onset.

Read the *shape* before the maximum. For a face/scrambled contrast the informative feature is the sharp rise beginning around 130–150 ms, in the N170 range; the curve then often stays elevated on a broad plateau where the exact argmax is close to arbitrary, so a sustained lift is far more trustworthy than an isolated spike.

Note also that cross-participant peaks tend to sit *later* than within-participant ones. The exact N170 topography is individual, and what transfers between heads is coarser and slower — which is precisely the gap Step 11's alignment is trying to close.

In [10]:
temporal_figure = plot_temporal_score_curve(
    temporal_scores[temporal_scores["Representation"].isin(REPRESENTATION_NAMES)],
    metric="balanced_accuracy",
    title=f"{' versus '.join(TARGET_NAMES)}: LOSO decoding",
    colors=REPRESENTATION_COLORS,
)
temporal_figure.add_hline(y=CHANCE_LEVEL, line_dash="dot", line_color="#777777")
temporal_figure.add_vline(x=0, line_color="#999999")
temporal_figure.update_yaxes(title_text="balanced accuracy")
temporal_figure.update_xaxes(title_text="time from image onset (s)")
temporal_figure.show()

In [11]:
peak_summary = temporal_scores.loc[
    temporal_scores.groupby("Representation")["Mean"].idxmax(),
    ["Representation", "Time", "Mean", "Std"],
].sort_values("Mean", ascending=False)
peak_summary = peak_summary.rename(columns={
    "Time": "peak_time_s",
    "Mean": "peak_balanced_accuracy",
    "Std": "peak_fold_std",
})
peak_summary.round(3)

,Representation,peak_time_s,peak_balanced_accuracy,peak_fold_std
1139,Aligned PCA (30C),0.340,0.619,0.048
112,Sensors,0.248,0.615,0.036
1389,"Aligned PCA (30C, calibration)",0.336,0.603,0.056
841,Aligned PCA (3C),0.152,0.586,0.044
361,Shared PCA (30),0.240,0.586,0.036
725,Unaligned PCA (30C),0.692,0.542,0.020


### Gain through time

The curves above are easy to over-read: several of them overlap and the eye is drawn to whichever happens to be on top at the maximum. The plot below re-expresses the same folds as a **paired difference against Sensors** at every latency — the mean and SEM across held-out participants of `(representation − Sensors)` balanced accuracy.

Pairing matters. Participants differ enormously in how decodable they are, and that between-participant variance dominates the ribbons in the previous figure while cancelling exactly in the difference. A curve sitting above zero is a sustained enhancement over raw sensors at that latency; below zero is a cost. This is the figure that answers "does the trajectory representation *add* anything, and when".

In [12]:
metric_rows = fold_scores[
    (fold_scores["Metric"] == "balanced_accuracy") & fold_scores["Time"].notna()
]
wide_scores = metric_rows.pivot_table(
    index=["Fold", "Time"], columns="Representation", values="Value"
)

gain_frames = []
for representation in OTHER_REPRESENTATIONS:
    gain = (wide_scores[representation] - wide_scores["Sensors"]).rename("Value").reset_index()
    summary = gain.groupby("Time")["Value"].agg(["mean", "std", "count"]).reset_index()
    summary["Model"] = representation
    summary["Metric"] = "balanced_accuracy_gain"
    summary["Mean"] = summary["mean"]
    summary["Std"] = summary["std"].fillna(0.0) / np.sqrt(summary["count"].clip(lower=1))
    gain_frames.append(summary[["Model", "Metric", "Time", "Mean", "Std"]])
gain_through_time = pd.concat(gain_frames, ignore_index=True)

gain_figure = plot_temporal_score_curve(
    gain_through_time,
    metric="balanced_accuracy_gain",
    title="Gain through time: representation minus Sensors (paired by held-out participant)",
    colors=REPRESENTATION_COLORS,
)
gain_figure.add_hline(y=0.0, line_dash="dot", line_color="#777777")
gain_figure.add_vline(x=0, line_color="#999999")
gain_figure.update_yaxes(title_text="balanced accuracy gain over Sensors")
gain_figure.update_xaxes(title_text="time from image onset (s)")
gain_figure.show()

## Step 9. Examine Held-Out-Participant Variability

A group mean can hide participants with qualitatively different decoding profiles, and with one fold per participant we can simply look at all of them. We retain one temporal curve per LOSO fold, labelled by the held-out participant, and reduce each fold to two predeclared post-onset scalars:

1. **Post-onset mean balanced accuracy:** average accuracy from 0 to 0.8 s.
2. **AUC above chance:** the temporal integral of `(balanced accuracy − chance)` over the same interval.

Representation-minus-Sensors differences stay paired within the same held-out participant. These are descriptive consistency diagnostics — the question "is this difference larger than chance?" is deferred to Step 10.

In [13]:
held_out_by_fold = split_audit.set_index("Fold")["held_out_subject"].to_dict()
fold_summary_records = []
for (representation, fold), rows in metric_rows.groupby(["Representation", "Fold"]):
    rows = rows.sort_values("Time")
    active_rows = rows[(rows["Time"] >= 0) & (rows["Time"] <= 0.8)]
    peak_index = rows["Value"].idxmax()
    fold_summary_records.append({
        "Representation": representation,
        "Fold": int(fold),
        "held_out_subject": held_out_by_fold[int(fold)],
        "peak_time_s": float(rows.loc[peak_index, "Time"]),
        "peak_balanced_accuracy": float(rows.loc[peak_index, "Value"]),
        "postonset_mean_balanced_accuracy": float(active_rows["Value"].mean()),
        "postonset_auc_above_chance": float(np.trapezoid(
            active_rows["Value"] - CHANCE_LEVEL, active_rows["Time"]
        )),
    })

fold_summary = pd.DataFrame(fold_summary_records)
representation_summary = (
    fold_summary.groupby("Representation")
    .agg(
        n_folds=("Fold", "nunique"),
        peak_ba_mean=("peak_balanced_accuracy", "mean"),
        peak_ba_std=("peak_balanced_accuracy", "std"),
        postonset_ba_mean=("postonset_mean_balanced_accuracy", "mean"),
        postonset_ba_std=("postonset_mean_balanced_accuracy", "std"),
        postonset_auc_mean=("postonset_auc_above_chance", "mean"),
        postonset_auc_std=("postonset_auc_above_chance", "std"),
    )
    .reset_index()
)
representation_summary.round(3)

,Representation,n_folds,peak_ba_mean,peak_ba_std,postonset_ba_mean,postonset_ba_std,postonset_auc_mean,postonset_auc_std
0,Aligned PCA (30C),3,0.651,0.038,0.560,0.025,0.048,0.020
1,"Aligned PCA (30C, calibration)",3,0.630,0.045,0.553,0.026,0.042,0.021
2,Aligned PCA (3C),3,0.614,0.031,0.487,0.017,-0.011,0.014
3,Sensors,3,0.621,0.035,0.523,0.008,0.018,0.006
4,Shared PCA (30),3,0.605,0.044,0.521,0.011,0.017,0.009
5,Unaligned PCA (30C),3,0.584,0.017,0.502,0.014,0.001,0.011


In [14]:
paired = fold_summary.pivot(
    index=["Fold", "held_out_subject"],
    columns="Representation",
    values=[
        "peak_balanced_accuracy",
        "postonset_mean_balanced_accuracy",
        "postonset_auc_above_chance",
    ],
)
paired_fold_differences = paired.index.to_frame(index=False)
for metric in (
    "peak_balanced_accuracy",
    "postonset_mean_balanced_accuracy",
    "postonset_auc_above_chance",
):
    for representation in OTHER_REPRESENTATIONS:
        slug = "".join(
            c if c.isalnum() else "_" for c in representation.lower()
        ).strip("_")
        paired_fold_differences[f"{metric}_{slug}_minus_sensors"] = (
            paired[(metric, representation)].to_numpy()
            - paired[(metric, "Sensors")].to_numpy()
        )
paired_fold_differences.round(3)

,Fold,held_out_subject,peak_balanced_accuracy_shared_pca__30_minus_sensors,peak_balanced_accuracy_unaligned_pca__30c_minus_sensors,peak_balanced_accuracy_aligned_pca__3c_minus_sensors,peak_balanced_accuracy_aligned_pca__30c_minus_sensors,postonset_mean_balanced_accuracy_shared_pca__30_minus_sensors,postonset_mean_balanced_accuracy_unaligned_pca__30c_minus_sensors,postonset_mean_balanced_accuracy_aligned_pca__3c_minus_sensors,postonset_mean_balanced_accuracy_aligned_pca__30c_minus_sensors,postonset_auc_above_chance_shared_pca__30_minus_sensors,postonset_auc_above_chance_unaligned_pca__30c_minus_sensors,postonset_auc_above_chance_aligned_pca__3c_minus_sensors,postonset_auc_above_chance_aligned_pca__30c_minus_sensors
0,0,01,-0.001,-0.087,-0.010,0.030,0.006,-0.040,-0.057,0.062,0.005,-0.032,-0.046,0.050
1,1,02,-0.016,-0.005,-0.003,0.024,-0.001,0.001,-0.026,0.031,-0.001,0.001,-0.021,0.025
2,2,03,-0.031,-0.018,-0.006,0.037,-0.010,-0.024,-0.024,0.018,-0.008,-0.019,-0.019,0.014


In [15]:
# One heatmap per representation, laid out as a row.
fold_panels = {}
for representation in REPRESENTATION_NAMES:
    rows = metric_rows[metric_rows["Representation"] == representation]
    matrix = rows.pivot(index="Fold", columns="Time", values="Value").sort_index()
    fold_panels[representation] = plot_heatmap(
        matrix.to_numpy(),
        x_labels=matrix.columns.to_numpy(dtype=float),
        y_labels=[f"sub-{held_out_by_fold[int(fold)]}" for fold in matrix.index],
        vmin=0,
        vmax=1,
        title=representation,
        xaxis_title="time from image onset (s)",
        yaxis_title="held-out participant",
        colorbar_label="BA",
    )

fold_heatmaps = facet_figures(
    fold_panels,
    n_cols=len(REPRESENTATION_NAMES),
    title="Balanced accuracy for every held-out participant",
    row_height=max(420, 32 * len(ANALYZED_SUBJECTS) + 200),
)
fold_heatmaps.show()


In [16]:
# Box + points per representation, one panel per predeclared summary.
distribution_panels = {}
for metric, pretty in (
    ("postonset_mean_balanced_accuracy", "Post-onset mean balanced accuracy"),
    ("postonset_auc_above_chance", "Post-onset AUC above chance"),
):
    distribution_panels[pretty] = plot_distribution_groups(
        groups=[
            fold_summary.loc[
                fold_summary["Representation"] == representation, metric
            ].to_numpy()
            for representation in REPRESENTATION_NAMES
        ],
        labels=REPRESENTATION_NAMES,
        color=[REPRESENTATION_COLORS[name] for name in REPRESENTATION_NAMES],
        show_points=True,
        baseline=0.0 if metric.endswith("auc_above_chance") else CHANCE_LEVEL,
        yaxis_title=pretty,
        title=pretty,
    )

fold_summary_figure = facet_figures(
    distribution_panels,
    n_cols=2,
    title="Per-fold distributions by representation",
    row_height=460,
    shared_yaxes=False,
)
fold_summary_figure.show()


The final view of the same folds puts one dot per held-out participant next to each representation's mean and standard error. This is the plot to consult when a group mean and a paired difference disagree: it shows immediately whether a representation wins on most participants or on one dramatic outlier.

In [17]:
representation_comparison_figure = plot_group_scatter_with_mean(
    [
        fold_summary.loc[
            fold_summary["Representation"] == representation, "peak_balanced_accuracy"
        ].to_numpy()
        for representation in REPRESENTATION_NAMES
    ],
    REPRESENTATION_NAMES,
    point_labels=[
        fold_summary.loc[
            fold_summary["Representation"] == representation, "held_out_subject"
        ].to_numpy()
        for representation in REPRESENTATION_NAMES
    ],
    title="Peak balanced accuracy by representation, one point per held-out participant",
    yaxis_title="peak balanced accuracy",
    baseline=CHANCE_LEVEL,
    baseline_label="chance",
    color=[REPRESENTATION_COLORS[name] for name in REPRESENTATION_NAMES],
    height=520,
)
representation_comparison_figure.show()

## Step 10. Statistical Significance

Everything so far has been descriptive. Now we ask whether the differences survive a null.

The test is a **paired permutation test** on the already-fitted predictions. Within each held-out participant, the predictions from the two representations are randomly swapped — that is the null hypothesis "the representation label is arbitrary" made concrete — and the observed balanced-accuracy difference is compared against the resulting distribution. Because the alignment is label-free it does not need to be rebuilt: only the classifiers' outputs are reshuffled.

Two features make this the right test here:

- **It is paired.** Swapping happens *within* a participant, so the enormous between-participant variance in decodability never enters the null.
- **It is corrected across time.** `temporal_correction='max_stat'` takes the maximum of each permutation's null across all latencies, so a single corrected p-value protects the entire time course rather than one hand-picked peak.

Three comparisons carry the argument:

| Comparison | Question |
|---|---|
| each representation vs **Sensors** | is this representation different from decoding raw channels? |
| Aligned PCA vs **Shared PCA** | does a *participant-specific* basis add anything to a shared one? |
| Aligned PCA vs **Unaligned PCA** | is it the **rotation**, or merely having a per-participant basis? |

The last row is the one that could explain the result away. If unaligned per-participant PCA already performed like the aligned version, the rotation would be decoration.

In [ ]:
stats_config = StatisticalAssessmentConfig(
    chance=ChanceAssessmentConfig(
        n_permutations=N_PERMUTATIONS, temporal_correction="max_stat"
    ),
    # The confidence interval's resampling budget; tied to the permutation count
    # so that a reduced run is cheap on both.
    n_bootstraps=N_PERMUTATIONS,
    unit_of_inference="group_mean",
    random_state=SEED,
    n_jobs=N_JOBS,
)
comparison_pairs = [
    *((representation, "Sensors") for representation in OTHER_REPRESENTATIONS),
    (ALIGNED_LARGE, SHARED_PCA),
    (ALIGNED_LARGE, UNALIGNED_PCA),
    (CALIBRATION, SHARED_PCA),
]

significance_frames = []
significance_figures = {}
for representation, baseline in comparison_pairs:
    comparison = run_paired_permutation_assessment(
        results[representation],
        results[baseline],
        "Logistic regression",
        "balanced_accuracy",
        stats_config,
    )
    comparison["Representation"] = representation
    comparison["Baseline"] = baseline
    significance_frames.append(comparison)
    key = f"{representation} vs {baseline}"
    significance_figures[key] = plot_temporal_statistical_assessment(
        comparison,
        title=f"{key}: paired permutation test ({N_PERMUTATIONS} shuffles, max-stat corrected)",
    )

significance_assessment = pd.concat(significance_frames, ignore_index=True)
significance_summary = (
    significance_assessment.loc[
        significance_assessment.groupby(["Representation", "Baseline"])["Observed"].idxmax()
    ]
    .reset_index(drop=True)
    .rename(columns={"Observed": "peak_observed_diff"})
)
significance_summary.round(4)

In [ ]:
for figure in significance_figures.values():
    figure.show()

<div class="alert alert-warning">
<b>⚠️ Read the p-value floor, not the decimal:</b><br>
A permutation p-value cannot go below <code>1 / (n_permutations + 1)</code>. With 200 shuffles that floor is 0.005, and a printed "0.00498" is the floor rather than an estimate of how small the true value is. Report such a result as <b>p &lt; 0.005</b>.
</div>

## Step 11. Bound the Alignment Result

A significant difference is not yet an interpretable one. Three further checks bound what any alignment gain can mean.

### 11a. The within-participant ceiling

LOSO asks a harder question than "can this signal be classified at all" — it demands that the pattern transfer between heads. Refitting the *identical* sliding decoder inside each participant with stratified k-fold cross-validation separates the two failure modes:

- if the within-participant curve is also low, the signal is weak;
- if the within-participant curve is high while LOSO is near chance, the signal is strong but **subject-specific**, and the entire cross-participant problem is one of correspondence rather than of signal-to-noise.

For faces the expected result is a large gap: within-participant decoding of face versus scrambled is comfortably above chance at the N170, while cross-participant decoding of the same trials is much weaker. That gap is the target alignment is aiming at, and this curve is the ceiling every representation is working toward.

In [ ]:
within_config = sensor_config.model_copy(deep=True)
within_config.cv = CVConfig(
    strategy="stratified", n_splits=WITHIN_SUBJECT_SPLITS, shuffle=True, random_state=SEED
)

within_frames = []
for subject in ANALYZED_SUBJECTS:
    rows = subject_ids == subject
    _, counts = np.unique(y[rows], return_counts=True)
    if len(counts) < 2 or counts.min() < WITHIN_SUBJECT_SPLITS:
        warnings.warn(f"Skipping {subject}: too few trials per class.", stacklevel=2)
        continue
    within_result = Experiment(within_config).run(
        X[rows], y[rows], sample_ids=trial_ids[rows],
        observation_level="epoch", time_axis=times,
    )
    curve = within_result.get_temporal_score_summary()
    curve["subject"] = subject
    within_frames.append(curve)

within_subject_scores = pd.concat(within_frames, ignore_index=True)
within_subject_scores = within_subject_scores[
    within_subject_scores["Metric"] == "balanced_accuracy"
]
within_subject_curve = (
    within_subject_scores.groupby("Time")["Mean"].agg(["mean", "std", "count"]).reset_index()
)
within_subject_curve["Model"] = "Within participant (sensors)"
within_subject_curve["Metric"] = "balanced_accuracy"
within_subject_curve["Mean"] = within_subject_curve["mean"]
within_subject_curve["Std"] = within_subject_curve["std"].fillna(0.0) / np.sqrt(
    within_subject_curve["count"].clip(lower=1)
)
within_subject_curve = within_subject_curve[["Model", "Metric", "Time", "Mean", "Std"]]

upper_bound_figure = plot_temporal_score_curve(
    pd.concat(
        [
            temporal_scores.loc[
                temporal_scores["Representation"] == "Sensors",
                ["Model", "Metric", "Time", "Mean", "Std"],
            ],
            within_subject_curve,
        ],
        ignore_index=True,
    ),
    metric="balanced_accuracy",
    title="Within-participant ceiling versus cross-participant transfer (sensors)",
    colors=REPRESENTATION_COLORS,
)
upper_bound_figure.add_hline(y=CHANCE_LEVEL, line_dash="dot", line_color="#777777")
upper_bound_figure.add_vline(x=0, line_color="#999999")
upper_bound_figure.update_yaxes(title_text="balanced accuracy")
upper_bound_figure.update_xaxes(title_text="time from image onset (s)")
upper_bound_figure.show()

within_subject_peaks = (
    within_subject_scores.loc[
        within_subject_scores.groupby("subject")["Mean"].idxmax(),
        ["subject", "Time", "Mean"],
    ]
    .rename(columns={"Time": "peak_time_s", "Mean": "peak_balanced_accuracy"})
    .sort_values("subject")
    .reset_index(drop=True)
)
within_subject_peaks.round(3)

### 11b. What the rotation actually does, geometrically

The decoder only reports *whether* alignment helped. Here we refit the alignment on every fold and read it out directly, with no classifier involved, using `TemporalProcrustesAlignment` on its own.

For each participant we record how well their grand-mean trajectory matched the training template **before** the rotation and **after** it, on a scale where 1 is a perfect match and 0 is orthogonal. The held-out participant is the row of interest: theirs is the mapping estimated without any labels, which is the one the decoding result depends on.

A large rise means the participant's principal axes pointed somewhere idiosyncratic and the rotation recovered the correspondence. A flat pair means either that the axes already agreed — in which case alignment has nothing to add — or that the grand-mean paths are too dissimilar in shape for any rotation to reconcile.

This is also where the alignment's known limitation becomes visible. The grand-mean path is dominated by the large early evoked response, so the rotation is fitted mostly to *that*. It should therefore align the axes carrying the face-versus-scrambled distinction well, and do little for a small late effect such as familiarity that lives elsewhere in the space.

To keep an $O(\text{participants}^2)$ diagnostic affordable we use up to `GEOMETRY_MAX_TRIALS` trials per participant; the grand-mean path is stable well below the full trial count.

In [ ]:
rng = np.random.default_rng(SEED)
keep = []
for subject in ANALYZED_SUBJECTS:
    rows = np.flatnonzero(subject_ids == subject)
    if len(rows) > GEOMETRY_MAX_TRIALS:
        rows = rng.choice(rows, size=GEOMETRY_MAX_TRIALS, replace=False)
    keep.append(rows)
keep = np.sort(np.concatenate(keep))
X_geometry, subjects_geometry = X[keep], subject_ids[keep]

geometry_records = []
for held_out in ANALYZED_SUBJECTS:
    train = subjects_geometry != held_out
    aligner = TemporalProcrustesAlignment(n_components=N_COMPONENTS, random_state=SEED)
    aligner.fit(X_geometry[train], groups=subjects_geometry[train])
    aligner.transform(X_geometry[~train], groups=subjects_geometry[~train])
    for subject, diagnostics in aligner.alignment_diagnostics_.items():
        geometry_records.append({
            "held_out_subject": held_out,
            "subject": subject,
            "role": "held out" if subject == held_out else "training",
            **diagnostics,
        })

alignment_geometry = pd.DataFrame(geometry_records).drop(columns=["seen_in_training"])
held_out_geometry = alignment_geometry[alignment_geometry["role"] == "held out"]

geometry_figure = plot_group_scatter_with_mean(
    [
        held_out_geometry["template_similarity_unrotated"].to_numpy(),
        held_out_geometry["template_similarity_rotated"].to_numpy(),
    ],
    ["before rotation", "after rotation"],
    point_labels=[
        held_out_geometry["subject"].to_numpy(),
        held_out_geometry["subject"].to_numpy(),
    ],
    title="Shape agreement between each held-out participant's mean path and the training template",
    yaxis_title="normalized similarity to template",
    baseline=0.0,
    color=[REPRESENTATION_COLORS[UNALIGNED_PCA], REPRESENTATION_COLORS[ALIGNED_LARGE]],
    height=460,
)
geometry_figure.show()

held_out_geometry.round(3)

### 11c. Is the gain just transduction?

Transductive alignment estimates the held-out participant's PCA and rotation from *all* of their unlabeled trials — including the very trials it is then scored on. No labels leak, but data does, and a sceptical reader is entitled to ask whether that is doing the work.

The calibration variant removes the concern by cross-fitting. Each half of the held-out participant's trials is mapped using the PCA and rotation estimated from the **other** half, so no trial ever informs the mapping applied to it, while every trial is still scored.

- If the transductive and calibration curves agree, the gain is genuine unsupervised domain adaptation from a calibration batch — a realistic requirement, since a new participant can always be shown some images before the labels matter.
- If the calibration curve collapses back onto Shared PCA, the apparent gain was the transduction.

In [ ]:
calibration_figure = plot_temporal_score_curve(
    temporal_scores[
        temporal_scores["Representation"].isin([SHARED_PCA, ALIGNED_LARGE, CALIBRATION])
    ],
    metric="balanced_accuracy",
    title="Transductive alignment versus calibration-half alignment",
    colors=REPRESENTATION_COLORS,
)
calibration_figure.add_hline(y=CHANCE_LEVEL, line_dash="dot", line_color="#777777")
calibration_figure.add_vline(x=0, line_color="#999999")
calibration_figure.update_yaxes(title_text="balanced accuracy")
calibration_figure.update_xaxes(title_text="time from image onset (s)")
calibration_figure.show()

peak_summary[peak_summary["Representation"].isin(
    [SHARED_PCA, ALIGNED_LARGE, CALIBRATION]
)].round(3)

## Step 13. Compare Contrasts Side by Side

The 12 steps above answer one question: **Famous versus Scrambled**. Several of the most useful claims, though, are *comparisons between* contrasts — whether familiarity behaves like face detection, whether Famous and Unfamiliar are interchangeable against Scrambled, and how a three-class target compares to the binaries. Those need all four panels in one view.

Only the **temporal decoding** is repeated for the other contrasts. The three validations in Step 11 bound the *alignment* claim rather than the comparison between contrasts, so they stay with the primary contrast. Even so this re-runs five representations three more times — set `MEG_CONTRAST_COMPARISON=0` to skip it during a quick local pass.


In [ ]:
COMPARISON_ACTIVE_WINDOW = (0.0, 0.8)  # post-onset interval, as in Step 9


def decode_contrast(conditions):
    """Decode one contrast across the five representations.

    Returns the fold-averaged time courses and the fold-level scores. Both are
    needed: the curves give the reported summaries, the folds give the spread
    across held-out participants that the figures draw.
    """
    container = _load_wakeman_henson_container(
        DERIVATIVES_ROOT,
        subjects=SUBJECTS,
        conditions=conditions,
        sensor_set=SENSOR_SET,
    )
    features = np.asarray(container.X, dtype=np.float32)
    time_axis = np.asarray(container.coords["time"], dtype=float)
    mapping = {value: index for index, value in enumerate(conditions)}
    target = np.array([mapping[value] for value in np.asarray(container.y, dtype=int)])
    groups = np.asarray(container.coords["subject"]).astype(str)
    ids = (
        np.asarray(container.ids).astype(str)
        if container.ids is not None
        else np.asarray([f"trial-{index}" for index in range(len(features))])
    )
    del container

    temporal_frames, fold_frames = [], []
    for representation, config in experiments.items():
        result = Experiment(config).run(
            features,
            target,
            groups=groups,
            sample_ids=ids,
            observation_level="epoch",
            inferential_unit="subject",
            time_axis=time_axis,
        )
        curve = result.get_temporal_score_summary()
        curve["Representation"] = representation
        curve["Model"] = representation
        temporal_frames.append(curve)

        folds = result.get_detailed_scores()
        folds["Representation"] = representation
        fold_frames.append(folds)
    return (
        pd.concat(temporal_frames, ignore_index=True),
        pd.concat(fold_frames, ignore_index=True),
    )


def contrast_fold_metrics(curves, folds, chance):
    """Per-fold values behind each reported summary, for one contrast.

    The peak is read off the **fold-averaged** curve and every fold is then
    sampled at that one latency, so the points average exactly to the reported
    number. Taking each fold's own maximum instead would select a different
    latency per fold and inflate the mean — the maximum of noisy estimates is
    not an estimate of the maximum.

    The two window summaries are averages over a fixed interval, so averaging
    over folds first or over time first gives the same value; the fold level is
    kept only for the spread.
    """
    rows = folds[(folds["Metric"] == "balanced_accuracy") & folds["Time"].notna()]
    records = []
    for representation, group in rows.groupby("Representation"):
        curve = curves[curves["Representation"] == representation].sort_values("Time")
        peak_time = float(curve.loc[curve["Mean"].idxmax(), "Time"])
        for fold, fold_rows in group.groupby("Fold"):
            fold_rows = fold_rows.sort_values("Time")
            at_peak = fold_rows.loc[
                (fold_rows["Time"] - peak_time).abs().idxmin(), "Value"
            ]
            active = fold_rows[
                (fold_rows["Time"] >= COMPARISON_ACTIVE_WINDOW[0])
                & (fold_rows["Time"] <= COMPARISON_ACTIVE_WINDOW[1])
            ]
            records.append(
                {
                    "Representation": representation,
                    "Fold": int(fold),
                    "peak_time_s": peak_time,
                    "peak_balanced_accuracy": float(at_peak),
                    "postonset_mean_balanced_accuracy": float(active["Value"].mean()),
                    "postonset_auc_above_chance": float(
                        np.trapezoid(active["Value"] - chance, active["Time"])
                    ),
                }
            )
    return pd.DataFrame(records)


COMPARISON_METRICS = {
    "peak_balanced_accuracy": "Peak balanced accuracy",
    "postonset_mean_balanced_accuracy": "Post-onset mean balanced accuracy",
    "postonset_auc_above_chance": "Post-onset AUC above chance",
}

contrast_panels = {}
contrast_folds = {}
contrast_scalars = []

if RUN_CONTRAST_COMPARISON:
    for conditions in COMPARISON_CONTRASTS:
        label = contrast_label(conditions, LABEL_NAMES, short=True)
        chance = 1.0 / len(conditions)
        if tuple(conditions) == tuple(CONDITIONS):
            curves, folds = temporal_scores, fold_scores  # already decoded above
        else:
            print(f"Decoding {label} ...")
            curves, folds = decode_contrast(tuple(conditions))
        curves = curves[curves["Representation"].isin(REPRESENTATION_NAMES)]
        folds = folds[folds["Representation"].isin(REPRESENTATION_NAMES)]

        panel = plot_temporal_score_curve(
            curves,
            metric="balanced_accuracy",
            title=label,
            colors=REPRESENTATION_COLORS,
        )
        panel.add_hline(y=chance, line_dash="dot", line_color="#777777")
        panel.add_vline(x=0, line_color="#999999")
        panel.update_yaxes(title_text="balanced accuracy")
        panel.update_xaxes(title_text="time from image onset (s)")
        contrast_panels[label] = panel

        per_fold = contrast_fold_metrics(curves, folds, chance)
        per_fold.insert(0, "Contrast", label)
        contrast_folds[label] = per_fold

        summary = per_fold.groupby("Representation", as_index=False)[
            ["peak_time_s", *COMPARISON_METRICS]
        ].mean()
        summary.insert(0, "Contrast", label)
        contrast_scalars.append(summary)

    # Chance differs between the binary and multiclass panels, so the y-axes
    # are left independent rather than shared.
    contrast_comparison_figure = facet_figures(
        contrast_panels,
        n_cols=2,
        title="LOSO decoding across contrasts",
        shared_yaxes=False,
    )
    contrast_comparison_figure.show()

    contrast_summary = pd.concat(contrast_scalars, ignore_index=True)
    contrast_fold_table = pd.concat(contrast_folds.values(), ignore_index=True)
    contrast_summary.to_csv(OUTPUT / "contrast_summary.csv", index=False)
    contrast_fold_table.to_csv(OUTPUT / "contrast_fold_metrics.csv", index=False)

    # One figure per scalar, in the same style as Step 9: a point per held-out
    # participant plus the mean and its SEM. Balanced accuracy and AUC keep
    # separate figures because they do not share a scale.
    contrast_metric_figures = {}
    for column, pretty in COMPARISON_METRICS.items():
        panels = {}
        for label, per_fold in contrast_folds.items():
            chance = 1.0 / len(
                next(c for c in COMPARISON_CONTRASTS
                     if contrast_label(c, LABEL_NAMES, short=True) == label)
            )
            panels[label] = plot_group_scatter_with_mean(
                [
                    per_fold.loc[
                        per_fold["Representation"] == representation, column
                    ].to_numpy()
                    for representation in REPRESENTATION_NAMES
                ],
                REPRESENTATION_NAMES,
                point_labels=[
                    per_fold.loc[
                        per_fold["Representation"] == representation, "Fold"
                    ].to_numpy()
                    for representation in REPRESENTATION_NAMES
                ],
                yaxis_title=pretty,
                baseline=0.0 if column.endswith("auc_above_chance") else chance,
                color=[REPRESENTATION_COLORS[name] for name in REPRESENTATION_NAMES],
            )
        figure = facet_figures(
            panels, n_cols=2, title=f"{pretty} across contrasts", shared_yaxes=False
        )
        figure.show()
        contrast_metric_figures[column] = figure

    for figure_name, figure in [
        ("contrast_comparison", contrast_comparison_figure),
        *((f"contrast_{column}", fig) for column, fig in contrast_metric_figures.items()),
    ]:
        figure.write_html(FIGURES_DIR / f"{figure_name}.html", include_plotlyjs="cdn")

    for column, pretty in COMPARISON_METRICS.items():
        print(f"\n{pretty}")
        print(
            contrast_summary.pivot(
                index="Contrast", columns="Representation", values=column
            )
            .round(3)
            .to_string()
        )
    display(contrast_summary.round(3))
else:
    print("Contrast comparison skipped (set MEG_CONTRAST_COMPARISON=1 to enable).")


## Conclusions & Interpretation Checklist

Before making a decoding claim from this analysis, verify the whole chain of evidence:

- **Participant separation:** every outer fold must have zero train/test participant overlap (Step 7's audit raises if not).
- **Fold-local preprocessing:** scaling, PCA and alignment are all fitted after the outer split — nothing that touched the held-out participant's data was fitted before it was set aside.
- **Inferential unit:** folds represent held-out participants, not independent time points or trials, and the permutation test swaps within participants for the same reason.
- **Temporal multiplicity:** a descriptive peak is not a corrected significance test; use the max-stat-corrected result from Step 10 and report the p-value floor honestly.
- **The rotation control:** any claim that alignment helps must be stated against **Unaligned PCA**, not only against Sensors.
- **Calibration scope:** aligned PCA consumes unlabeled held-out-participant trials and is transductive; the Step 11c variant is the version that can be described as leakage-free.
- **The comparison is against an *unaligned* sensor decoder.** Euclidean Alignment and Riemannian Procrustes Analysis are genuine cross-participant sensor-space alignment methods. Until they run as baselines on these same folds, the defensible claim is narrow: aligning a time-resolved trajectory *shape* has no sensor-space equivalent, because the matched object is a path rather than a distribution.
- **Sample size.** With sixteen participants, the permutation test controls the label mapping, not the sample.

<div class="alert alert-success">
<b>🎯 Main takeaway:</b><br>
At single-trial decoding, a 30-component trajectory does not beat 306 sensors — and cannot, since it is a rotation and truncation of them. Matching them <i>is</i> the compression result: all the discriminative information in a tenth of the dimensions. The claim that only the low-dimensional framing supports is the alignment one, and it must be read against the unaligned control, the within-participant ceiling, and the calibration variant rather than against sensors alone.
</div>

## Running the Same Analysis Headlessly

The companion script repeats the same computations, exports the complete `ExperimentResult` objects, and invokes the same report renderer:

```bash
python scripts/analysis_megfaces_decoding.py \
    --derivatives-root <prepared-derivatives> \
    --output outputs/megfaces_decoding
```

Useful variations:

```bash
# quick validation run
python scripts/analysis_megfaces_decoding.py --subjects 01 02 03 --n-permutations 20

# familiarity rather than perceptual category
python scripts/analysis_megfaces_decoding.py --conditions 1 2

# three-class decoding, chance 1/3
python scripts/analysis_megfaces_decoding.py --conditions 1 2 3

# all four contrasts as one sweep: each lands in its own subdirectory and every
# report figure gains one panel per contrast
python scripts/analysis_megfaces_decoding.py --contrasts 1-3 2-3 1-2 1-2-3

# combine contrasts that were run as separate cluster jobs, without decoding
python scripts/analysis_megfaces_decoding.py --report-only

# restrict to one helmet sensor selection
python scripts/analysis_megfaces_decoding.py --sensor-set occipital
```

Condition ids are `1 = Famous`, `2 = Unfamiliar`, `3 = Scrambled`. Two ids give binary decoding; three give multiclass, and chance moves accordingly.

<div class="alert alert-info">
<b>💡 A contrast worth running:</b><br>
Repeat the analysis with <code>--conditions 1 2</code> (famous versus unfamiliar). Familiarity has no N170 — it sits at chance through 0.2 s and then climbs slowly, still rising when the epoch ends. Alignment should help far less there, because the grand-mean path it rotates onto is dominated by the early evoked response that familiarity does not modulate. The two contrasts together are what make the alignment story interpretable rather than merely positive.
</div>